# 📦 Netelpro: Fase 2 - Compilación del Corpus Masivo (250-500M tokens)
### *Cosmopedia v2 + OpenAssistant/Alpaca-ES + FineWeb-2 Español + Código de Sistemas + Lore Netelpro*

Este notebook **solo construye el corpus** — no entrena ningún modelo. Genera shards binarios
packed `uint16` (mismo formato que usa `train_teo_v2.py` y el nuevo `train_teo_sds.py`) y los
empaqueta al final para subirlos como **Kaggle Dataset privado**, reutilizable en cualquier
notebook de entrenamiento posterior sin re-descargar de HuggingFace.

---
### ⚙️ Configuración en Kaggle:
- **Accelerator:** CPU o GPU (no importa, este notebook no entrena — GPU T4 ayuda solo por más RAM/CPU asignada)
- **Internet:** **On** (obligatorio — streaming desde HuggingFace)
- **Persistence:** Files on (para que el corpus sobreviva si la sesión se corta y hay que reanudar)


## 1. Clonar/Actualizar Repositorio Netelpro (privado) e Instalar Dependencias
Repo privado: usa **Kaggle Secrets** (`Add-ons` -> `Secrets` -> agregá un secret llamado `GITHUB_TOKEN`
con tu Personal Access Token de GitHub). Si no configurás el secret, pegá el token manualmente
en la variable `GITHUB_TOKEN` de la celda de abajo antes de correrla (y no subas/compartas el
notebook con el token pegado adentro).

In [ ]:
%cd /kaggle/working
import os

GITHUB_TOKEN = ""  # <-- si no usás Kaggle Secrets, pegá tu token acá

if not GITHUB_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = ""

REPO_URL = (
    f"https://{GITHUB_TOKEN}@github.com/jona2428/netelpro.git"
    if GITHUB_TOKEN
    else "https://github.com/jona2428/netelpro.git"
)

if not os.path.exists('netelpro'):
    !git clone {REPO_URL}
else:
    !cd /kaggle/working/netelpro && git pull origin master
%cd /kaggle/working/netelpro

!pip install -q tokenizers datasets pyarrow requests numpy


## 2. (Opcional) Reanudar desde un Corpus Parcial Anterior
Si esta es la continuación de una sesión de Kaggle previa que se cortó a mitad de camino,
subí el `.tar.gz` empaquetado en la celda final de esa sesión como Kaggle Dataset
(ej: `teo-v2-massive-corpus-partial`) y montalo acá antes de correr la Celda 4.
El builder detecta `dedup_state.json` y los shards ya completos, y sigue desde ahí
(`--resume` está activo por default).

In [ ]:
import os
os.makedirs('data/teo_v2_massive', exist_ok=True)

# Descomentar y ajustar el nombre del dataset si estás reanudando:
# !tar -xzvf /kaggle/input/teo-v2-massive-corpus-partial/massive_corpus_partial.tar.gz -C data/teo_v2_massive

print('📂 Contenido actual de data/teo_v2_massive:', os.listdir('data/teo_v2_massive') if os.path.exists('data/teo_v2_massive') else 'vacío')


## 3. Compilar Corpus Masivo (20 shards × 25M tokens = 500M tokens)
Streaming simultáneo de 5 dominios (razonamiento/lore, conversaciones, Cosmopedia STEM, código
de sistemas, filosofía/ciencia en español), deduplicado, con checkpoint de progreso cada shard
completo (`dedup_state.json` + shards ya cerrados). Si la sesión de Kaggle se corta, volvé a
correr esta misma celda — retoma solo. Ajustá `--target-shards` a 10 (250M) si preferís una
corrida más corta primero.

In [ ]:
!python training/data/build_massive_corpus.py \
    --out-dir data/teo_v2_massive \
    --tokenizer-path data/teo_v2/tokenizer.json \
    --target-shards 20 \
    --shard-size 25000000 \
    --persona-multiplier 100


## 4. Verificación del Corpus Compilado

In [ ]:
import sys
sys.path.insert(0, '.')
from pathlib import Path
from training.data.compile_packed import load_stream, stream_stats

shard_dir = Path('data/teo_v2_massive')
shards = sorted(shard_dir.glob('shard_*.bin'))
print(f'📂 Shards completos: {len(shards)}')

total_tokens = 0
for s in shards:
    stream = load_stream(s)
    stats = stream_stats(stream)
    total_tokens += stats['total_tokens']
    assert stats['pad_fraction'] == 0.0, f'{s.name} tiene padding! {stats}'
    print(f"  {s.name}: {stats['total_tokens']:,} tokens, pad_fraction={stats['pad_fraction']}")

print(f'\n🎉 Total: {total_tokens:,} tokens en {len(shards)} shard(s) — CERO padding verificado.')


## 5. Empaquetar Corpus para Reutilización (Kaggle Dataset)
Comprime todos los shards + tokenizer + metadata. Si el corpus quedó incompleto (sesión
cortada por límite de tiempo), este mismo archivo sirve para retomar en la Celda 2 de la
próxima sesión. Si quedó completo, subilo como Dataset (ej. `teo-v2-massive-corpus`) para
usarlo directamente en `train_teo_sds.py` sin repetir la descarga.

In [ ]:
!tar -czvf /kaggle/working/massive_corpus_partial.tar.gz -C data teo_v2_massive
print('🎉 Corpus empaquetado en /kaggle/working/massive_corpus_partial.tar.gz')
print('   Subilo como Kaggle Dataset privado (ej: teo-v2-massive-corpus) para usarlo')
print('   en el notebook de entrenamiento de Teo-SDS (train_teo_sds.py) sin re-descargar.')
